# D3 후속: KURE-v1 · bge-m3 임베딩 (Colab GPU)
1. 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택
2. 아래 셀을 위에서부터 차례로 실행
3. 업로드할 파일: `colab_input.json` (로컬 `data/processed/`에 있음)
4. 마지막 셀이 `colab_output.zip`을 내려받게 한다 → 로컬 `data/processed/`에 풀기

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU 없음 - 런타임 유형을 확인하세요")

In [ ]:
from google.colab import files
up = files.upload()   # colab_input.json 선택
assert "colab_input.json" in up

In [ ]:
import json, time, numpy as np
from sentence_transformers import SentenceTransformer
src = json.load(open("colab_input.json", encoding="utf-8"))
print(len(src["chunks"]), "청크,", len(src["queries"]), "질문")
MODELS = ["nlpai-lab/KURE-v1", "BAAI/bge-m3"]
MAX_LEN, BATCH = 512, 32   # 로컬과 같은 최대 길이. 청크는 길이순으로 묶어 패딩을 줄인다

def encode(model, texts):
    order = np.argsort([len(t) for t in texts])
    vecs = model.encode([texts[i] for i in order], batch_size=BATCH, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
    out = np.empty_like(vecs)
    out[order] = vecs
    return out.astype(np.float32)

for name in MODELS:
    short = name.split("/")[-1].lower()
    t0 = time.time()
    m = SentenceTransformer(name, device="cuda")
    m.max_seq_length = MAX_LEN
    m.half()   # fp16: GPU에서 속도 2배 안팎. 코사인 유사도 차이는 매우 작다
    ch = encode(m, src["chunks"])
    qu = encode(m, src["queries"])
    np.savez(f"colab_output_{short}.npz", chunks=ch, queries=qu)
    print(name, ch.shape, qu.shape, f"{time.time() - t0:.0f}초", "NaN:", bool(np.isnan(ch).any()))
    del m; torch.cuda.empty_cache()

In [ ]:
!zip -q colab_output.zip colab_output_*.npz
files.download("colab_output.zip")